# Datenqualitätsprüfung der Web-Scraping-Erhebung

Dieses Notebook untersucht die technische Vollständigkeit der automatisierten Datenerhebung für **Tennistown**, **Tennis-Heine** und **Tennis-Point** im Zeitraum vom **14.06.2026 bis zum 17.07.2026**.

Dazu werden:

- die CSV-Rohdateien des Verzeichnisses `data/raw` eingelesen
- Händler, Datum und Abrufkennung aus den Dateinamen extrahiert
- die Anzahl der erfassten Produktbeobachtungen je Datei bestimmt
- die tatsächlich vorhandenen Dateien mit den erwarteten täglichen Abrufen (`run1` und `run2`) abgeglichen
- die Itemanzahlen in einer Datenqualitätsmatrix gegenübergestellt
- ergänzende Informationen aus den GitHub-Actions-Logs für Tennis-Point dokumentiert

Die Ergebnisse dienen dazu, die Zuverlässigkeit der Datenerhebung einzuschätzen und die Datengrundlage für die nachfolgenden Aufbereitungs- und Analyseschritte zu bewerten.

> **Hinweis:** In diesem Notebook erfolgt noch keine inhaltliche Bereinigung oder Analyse der Preis- und Verfügbarkeitsdaten.

In [73]:
import os
import re
import pandas as pd

In [74]:
# --- PFAD- UND ZEITRAUM-KONFIGURATION ---
# Verzeichnis mit den unveränderten CSV-Rohdateien
data_raw_dir = '../data/raw'

# Untersuchungszeitraum
start_date = '2026-06-14'
end_date = '2026-07-17'

# Reguläre automatisierte Abrufe
regular_runs = ['run1', 'run2']

# Untersuchte Händler 
examined_retailers = ['tennistown', 'tennis_heine', 'tennis_point']

# --- DATEINAMEN-KONFIGURATION ---
# Regulärer Ausdruck zur Zerlegung der CSV-Dateinamen
# Gruppe 1: Händler
# Gruppe 2: Datum im Format YYYY-MM-DD
# Gruppe 3: Abrufkennung (run1, run2 oder manual_XXh)
file_pattern = re.compile(r'^(tennistown|tennis_heine|tennis_point)_?(\d{4}-\d{2}-\d{2})_?(run1|run2|manual_\d{1,2}h)\.csv$')

In [75]:
# --- FUNKTION ZUR BESTIMMUNG DER ITEM-ANZAHL ---
def count_csv_rows(file_path):
    '''
    Liest eine CSV-Datei ein und bestimmt die Anzahl der enthaltenen
    Produktbeobachtungen ohne die Header-Zeile.
    Eine vollständig leere CSV-Datei wird als Datei mit 0 Items gewertet.
    Bei anderen Einlesefehlern werden der Fehlerstatus und die
    Fehlermeldung zurückgegeben.
    '''

    try:
        df_file = pd.read_csv(file_path, encoding='utf-8')

        return len(df_file), None

    except pd.errors.EmptyDataError:
        return 0, None

    except Exception as error:
        return None, str(error)

In [76]:
# --- SCAN DER CSV-ROHDATEN ---
data_raw_list = []

if not os.path.exists(data_raw_dir):
    raise FileNotFoundError(f'Der Pfad "{data_raw_dir}" wurde nicht gefunden.')

# Alle Dateien des Rohdatenverzeichnisses durchsuchen
for file_name in sorted(os.listdir(data_raw_dir)):

    # Nur CSV-Dateien untersuchen
    if not file_name.endswith('.csv'):
        continue

    # Dateinamen anhand des festgelegten Musters zerlegen
    match = file_pattern.match(file_name)

    # Nicht passende Dateien überspringen und zur Kontrolle ausgeben
    if not match:
        print(f'Dateiname entspricht nicht dem erwarteten Muster: {file_name}')
        continue

    retailer, date_str, run_tag = match.groups()

    file_path = os.path.join(data_raw_dir, file_name)

    # Anzahl der gespeicherten Produktbeobachtungen bestimmen
    row_count, read_error = count_csv_rows(file_path)

    data_raw_list.append({
        'Date': date_str,
        'Retailer': retailer,
        'Run tag': run_tag,
        'File name': file_name,
        'File exists': True,
        'Items': row_count,
        'Read error': read_error
    })

# Tabellarische Laufübersicht aus allen vorhandenen Dateien erzeugen
df_run_files = pd.DataFrame(data_raw_list)

print(f'{len(df_run_files)} CSV-Dateien wurden erkannt und ausgewertet.')

171 CSV-Dateien wurden erkannt und ausgewertet.


In [77]:
# --- VOLLSTÄNDIGES ERWARTETES ABRUFGITTER ERZEUGEN ---
# Vollständige Liste aller Tage des Untersuchungszeitraums
complete_date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Alle erwarteten Kombinationen aus Datum, Händler und Abruf erzeugen
expected_runs_index = pd.MultiIndex.from_product(
    [
        complete_date_range,
        examined_retailers,
        regular_runs
    ],
    names=[
        'Date',
        'Retailer',
        'Run tag'
    ]
)

df_expected_runs = (expected_runs_index.to_frame(index=False))

# Datum der tatsächlich vorhandenen Dateien ebenfalls konvertieren
if not df_run_files.empty:
    df_run_files['Date'] = pd.to_datetime(df_run_files['Date'])

# Erwartete Abrufe mit den tatsächlich vorhandenen Dateien verbinden
df_run_overview = df_expected_runs.merge(
    df_run_files,
    on=['Date', 'Retailer', 'Run tag'],
    how='left'
)

# Fehlende Dateiangaben eindeutig kennzeichnen
df_run_overview['File exists'] = (df_run_overview['File exists'].fillna(False).astype(bool))

# Itemanzahl als Nullable-Integer speichern
# Fehlende Datei: Items = <NA>
# Vorhandene, aber leere Datei: Items = 0
df_run_overview['Items'] = (df_run_overview['Items'].astype('Int64'))

# Nach Datum, Händler und Abruf sortieren
df_run_overview = (
    df_run_overview
    .sort_values([
        'Date',
        'Retailer',
        'Run tag'
    ])
    .reset_index(drop=True)
)

display(df_run_overview.head(15))

,Date,Retailer,Run tag,File name,File exists,Items,Read error
0,2026-06-14,tennis_heine,run1,tennis_heine_2026-06-14_run1.csv,True,484,None
1,2026-06-14,tennis_heine,run2,tennis_heine_2026-06-14_run2.csv,True,484,None
2,2026-06-14,tennis_point,run1,tennis_point_2026-06-14_run1.csv,True,649,None
3,2026-06-14,tennis_point,run2,tennis_point_2026-06-14_run2.csv,True,633,None
4,2026-06-14,tennistown,run1,tennistown_2026-06-14_run1.csv,True,709,None
5,2026-06-14,tennistown,run2,tennistown_2026-06-14_run2.csv,True,707,None
6,2026-06-15,tennis_heine,run1,tennis_heine_2026-06-15_run1.csv,True,484,None
7,2026-06-15,tennis_heine,run2,tennis_heine_2026-06-15_run2.csv,True,484,None
8,2026-06-15,tennis_point,run1,tennis_point_2026-06-15_run1.csv,True,0,None
9,2026-06-15,tennis_point,run2,tennis_point_2026-06-15_run2.csv,True,648,None


In [78]:
# --- DATENQUALITÄTS-MATRIX ERZEUGEN ---
# Technischen Spaltennamen aus Händler und Abruf bilden
df_run_overview['Retailer_Run'] = (df_run_overview['Retailer'] + '_' + df_run_overview['Run tag'])

# Matrix mit der Anzahl erfasster Items erzeugen
df_matrix = df_run_overview.pivot(index='Date', columns='Retailer_Run', values='Items')

# Hilfsspalte nach der Matrixerzeugung wieder entfernen
df_run_overview = df_run_overview.drop(columns=['Retailer_Run'])

# Gewünschte Spaltenreihenfolge festlegen
matrix_columns = [
    'tennistown_run1',
    'tennistown_run2',
    'tennis_heine_run1',
    'tennis_heine_run2',
    'tennis_point_run1',
    'tennis_point_run2'
]

df_matrix = df_matrix.reindex(columns=matrix_columns)

print('--- DATENQUALITÄTS-MATRIX ---')
print('Zeigt die Anzahl erfasster Produktbeobachtungen je Händler und automatisiertem Abruf an.\n')

display(df_matrix)

--- DATENQUALITÄTS-MATRIX ---
Zeigt die Anzahl erfasster Produktbeobachtungen je Händler und automatisiertem Abruf an.



Retailer_Run,tennistown_run1,tennistown_run2,tennis_heine_run1,tennis_heine_run2,tennis_point_run1,tennis_point_run2
Date,,,,,,
2026-06-14,709,707,484,484,649,633
2026-06-15,706,705,484,484,0,648
2026-06-16,703,706,484,484,242,661
2026-06-17,705,706,484,484,0,602
2026-06-18,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2026-06-19,705,706,484,484,225,634
2026-06-20,706,705,484,484,0,653
2026-06-21,705,705,484,484,341,288
2026-06-22,706,706,484,484,135,641


In [79]:
# --- MANUELL ERFASSTE WORKFLOW-INFORMATIONEN FÜR TENNIS-POINT ---
# Die Anzahl dauerhaft fehlgeschlagener Requests wurde aus den GitHub-Actions-Logs der jeweiligen Tennis-Point-Läufe übernommen

# None bedeutet: Für diesen Lauf liegt keine Angabe aus dem Log vor
# --> Die Prüfung erfolgte nur bei denjenigen Durchläufen, bei denen überprüft werden musste, ob der Abruf vollständig war

# 0 bedeutet: Der Lauf wurde ohne dauerhaft fehlgeschlagene Requests beendet

tennis_point_quality_data = [
    {
        'Date': '2026-06-14',
        'Run': 'run1',
        'Items': 649,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-06-14',
        'Run': 'run2',
        'Items': 633,
        'Retry_Max_Reached': 20
    },
    {
        'Date': '2026-06-15',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-15',
        'Run': 'run2',
        'Items': 648,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-06-16',
        'Run': 'run1',
        'Items': 242,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-16',
        'Run': 'run2',
        'Items': 661,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-06-17',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-17',
        'Run': 'run2',
        'Items': 602,
        'Retry_Max_Reached': 78
    },
    {
        'Date': '2026-06-18',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-18',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-19',
        'Run': 'run1',
        'Items': 225,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-19',
        'Run': 'run2',
        'Items': 634,
        'Retry_Max_Reached': 35
    },
    {
        'Date': '2026-06-20',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-20',
        'Run': 'run2',
        'Items': 653,
        'Retry_Max_Reached': 21
    },
    {
        'Date': '2026-06-21',
        'Run': 'run1',
        'Items': 341,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-21',
        'Run': 'run2',
        'Items': 288,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-22',
        'Run': 'run1',
        'Items': 135,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-22',
        'Run': 'run2',
        'Items': 641,
        'Retry_Max_Reached': 8
    },
    {
        'Date': '2026-06-23',
        'Run': 'run1',
        'Items': 453,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-23',
        'Run': 'run2',
        'Items': 299,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-24',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-24',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-25',
        'Run': 'run1',
        'Items': 329,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-25',
        'Run': 'run2',
        'Items': 40,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-26',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-26',
        'Run': 'run2',
        'Items': 499,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-27',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-27',
        'Run': 'run2',
        'Items': 599,
        'Retry_Max_Reached': 30
    },
    {
        'Date': '2026-06-28',
        'Run': 'run1',
        'Items': 609,
        'Retry_Max_Reached': 125
    },
    {
        'Date': '2026-06-28',
        'Run': 'run2',
        'Items': 622,
        'Retry_Max_Reached': 33
    },
    {
        'Date': '2026-06-29',
        'Run': 'run1',
        'Items': 245,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-29',
        'Run': 'run2',
        'Items': 320,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-06-30',
        'Run': 'run1',
        'Items': 593,
        'Retry_Max_Reached': 144
    },
    {
        'Date': '2026-06-30',
        'Run': 'run2',
        'Items': 579,
        'Retry_Max_Reached': 140
    },
    {
        'Date': '2026-07-01',
        'Run': 'run1',
        'Items': 623,
        'Retry_Max_Reached': 72
    },
    {
        'Date': '2026-07-01',
        'Run': 'run2',
        'Items': 554,
        'Retry_Max_Reached': 117
    },
    {
        'Date': '2026-07-02',
        'Run': 'run1',
        'Items': 609,
        'Retry_Max_Reached': 180
    },
    {
        'Date': '2026-07-02',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-03',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-03',
        'Run': 'run2',
        'Items': 623,
        'Retry_Max_Reached': 47
    },
    {
        'Date': '2026-07-04',
        'Run': 'run1',
        'Items': 632,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-07-04',
        'Run': 'run2',
        'Items': 632,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-07-05',
        'Run': 'run1',
        'Items': 632,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-07-05',
        'Run': 'run2',
        'Items': 632,
        'Retry_Max_Reached': 2
    },
    {
        'Date': '2026-07-06',
        'Run': 'run1',
        'Items': 620,
        'Retry_Max_Reached': 142
    },
    {
        'Date': '2026-07-06',
        'Run': 'run2',
        'Items': 622,
        'Retry_Max_Reached': 71
    },
    {
        'Date': '2026-07-07',
        'Run': 'run1',
        'Items': 516,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-07',
        'Run': 'run2',
        'Items': 632,
        'Retry_Max_Reached': 0
    },
    {
        'Date': '2026-07-08',
        'Run': 'run1',
        'Items': 628,
        'Retry_Max_Reached': 70
    },
    {
        'Date': '2026-07-08',
        'Run': 'run2',
        'Items': 600,
        'Retry_Max_Reached': 10
    },
    {
        'Date': '2026-07-09',
        'Run': 'run1',
        'Items': 469,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-09',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-10',
        'Run': 'run1',
        'Items': 601,
        'Retry_Max_Reached': 70
    },
    {
        'Date': '2026-07-10',
        'Run': 'run2',
        'Items': 488,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-11',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-11',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-12',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-12',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-13',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-13',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-14',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-14',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-15',
        'Run': 'run1',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-15',
        'Run': 'run2',
        'Items': 494,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-16',
        'Run': 'run1',
        'Items': 241,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-16',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    },
    {
        'Date': '2026-07-17',
        'Run': 'run1',
        'Items': 562,
        'Retry_Max_Reached': 158
    },
    {
        'Date': '2026-07-17',
        'Run': 'run2',
        'Items': 0,
        'Retry_Max_Reached': None
    }
]

In [80]:
# DataFrame aus den manuell erfassten Workflow-Informationen erzeugen
df_tennis_point_logs = pd.DataFrame(tennis_point_quality_data)

# Datum in ein echtes Datumsformat umwandeln
df_tennis_point_logs['Date'] = pd.to_datetime(df_tennis_point_logs['Date'])

# Spaltennamen an die übrige Laufübersicht anpassen
df_tennis_point_logs = df_tennis_point_logs.rename(
    columns={
        'Run': 'Run tag',
        'Retry_Max_Reached': 'Retry max reached requests'
    }
)

# Retry-Spalte als Integer-Spalte mit erlaubten fehlenden Werten speichern
df_tennis_point_logs['Retry max reached requests'] = (df_tennis_point_logs['Retry max reached requests'].astype('Int64'))

# Manuelle Item-Spalte entfernen
# Die Itemanzahl wird bereits automatisch aus den CSV-Dateien bestimmt
df_tennis_point_logs = df_tennis_point_logs.drop(columns=['Items'], errors='ignore')

# Händler als feste Metadaten ergänzen
df_tennis_point_logs['Retailer'] = 'tennis_point'

# Nach Datum und Abruf sortieren
df_tennis_point_logs = (
    df_tennis_point_logs
    .sort_values([
        'Date',
        'Run tag'
    ])
    .reset_index(drop=True)
)

display(df_tennis_point_logs.head())

,Date,Run tag,Retry max reached requests,Retailer
0,2026-06-14,run1,0,tennis_point
1,2026-06-14,run2,20,tennis_point
2,2026-06-15,run1,<NA>,tennis_point
3,2026-06-15,run2,0,tennis_point
4,2026-06-16,run1,<NA>,tennis_point


In [81]:
# --- ERGÄNZUNG DER TENNIS-POINT-WORKFLOW-INFORMATIONEN ---
df_run_quality = df_run_overview.merge(
    df_tennis_point_logs,
    on=[
        'Date',
        'Retailer',
        'Run tag'
    ],
    how='left'
)

display(df_run_quality[df_run_quality['Retailer'] == 'tennis_point'].head(10))

,Date,Retailer,Run tag,File name,File exists,Items,Read error,Retry max reached requests
2,2026-06-14,tennis_point,run1,tennis_point_2026-06-14_run1.csv,True,649,None,0
3,2026-06-14,tennis_point,run2,tennis_point_2026-06-14_run2.csv,True,633,None,20
8,2026-06-15,tennis_point,run1,tennis_point_2026-06-15_run1.csv,True,0,None,<NA>
9,2026-06-15,tennis_point,run2,tennis_point_2026-06-15_run2.csv,True,648,None,0
14,2026-06-16,tennis_point,run1,tennis_point_2026-06-16_run1.csv,True,242,None,<NA>
15,2026-06-16,tennis_point,run2,tennis_point_2026-06-16_run2.csv,True,661,None,0
20,2026-06-17,tennis_point,run1,tennis_point_2026-06-17_run1.csv,True,0,None,<NA>
21,2026-06-17,tennis_point,run2,tennis_point_2026-06-17_run2.csv,True,602,None,78
26,2026-06-18,tennis_point,run1,NaN,False,<NA>,NaN,<NA>
27,2026-06-18,tennis_point,run2,NaN,False,<NA>,NaN,<NA>
